# 02d — Strategy D: SegFormer-B0 fine-tune (non-GAN baseline)

Fine-tune [`nvidia/mit-b0`](https://huggingface.co/nvidia/mit-b0) + a 2-class semantic-segmentation head on our synthetic pairs, with **BCE + Dice loss**. No adversarial training — this is the control for whether the GAN in strategies A/B is actually doing real work.

Prereqs: `pip install torch torchvision transformers`

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR))
import dstroke_utils as d

REPO_ROOT = Path('C:/Users/MichelleJacobs/Repos/AIAi')
SYNTH = REPO_ROOT / 'data_notcommitted' / 'dstroke_synth'
OUT = REPO_ROOT / 'AIA - AI Tool - Fingerprinting-1' / 'outputs' / 'dstroke'
OUT.mkdir(parents=True, exist_ok=True)
CKPT = OUT / 'strategy_D_segformer.pt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
MODEL_NAME = 'nvidia/mit-b0'
model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'bg', 1: 'edge'},
    label2id={'bg': 0, 'edge': 1},
    ignore_mismatched_sizes=True,
).to(DEVICE)
proc = SegformerImageProcessor(do_resize=False, do_reduce_labels=False)
print('SegFormer loaded,', sum(p.numel() for p in model.parameters())/1e6, 'M params')

In [ ]:
class SegDS(Dataset):
    def __init__(self, root: Path, augment: bool = False):
        self.root = Path(root)
        self.paintings = sorted((self.root / 'paintings').glob('*.png'))
        self.augment = augment
    def __len__(self): return len(self.paintings)
    def __getitem__(self, i):
        p = self.paintings[i]
        e = self.root / 'edges' / p.name
        rgb = np.array(Image.open(p).convert('RGB'), dtype=np.float32) / 255.0
        edge = np.array(Image.open(e).convert('L'),   dtype=np.float32) / 255.0
        if self.augment and np.random.rand() < 0.5:
            rgb = rgb[:, ::-1].copy(); edge = edge[:, ::-1].copy()
        enc = proc(images=(rgb*255).astype(np.uint8), return_tensors='pt')
        x = enc['pixel_values'][0]                         # (3, H, W)
        y = torch.from_numpy((edge > 0.5).astype(np.float32)).unsqueeze(0)  # (1, H, W)
        return x.float(), y.float()

train_ds = SegDS(SYNTH / 'train', augment=True)
val_ds   = SegDS(SYNTH / 'val',   augment=False)
print(len(train_ds), len(val_ds))
train_dl = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=0)

In [ ]:
def dice_loss(logits_edge, target):
    probs = torch.sigmoid(logits_edge)
    inter = (probs * target).sum(dim=(-1, -2))
    denom = probs.sum(dim=(-1, -2)) + target.sum(dim=(-1, -2)) + 1e-6
    return 1 - (2 * inter / denom).mean()

bce = nn.BCEWithLogitsLoss()
LR = 6e-5; EPOCHS = 30
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

history_D = {'train': [], 'val_iou': [], 'val_f1': []}

for epoch in range(1, EPOCHS+1):
    model.train(); loss_ep = 0.0
    for x, y in tqdm(train_dl, desc=f'epoch {epoch}/{EPOCHS}'):
        x=x.to(DEVICE); y=y.to(DEVICE)
        logits = model(pixel_values=x).logits                                 # (B, 2, h, w)
        logits = F.interpolate(logits, size=y.shape[-2:], mode='bilinear', align_corners=False)
        edge_logit = logits[:, 1:2] - logits[:, 0:1]                          # binary logit
        loss = bce(edge_logit, y) + dice_loss(edge_logit, y)
        opt.zero_grad(); loss.backward(); opt.step()
        loss_ep += float(loss)
    train_loss = loss_ep / len(train_dl)

    model.eval(); ious, f1s = [], []
    with torch.no_grad():
        for x, y in val_dl:
            x=x.to(DEVICE); y=y.to(DEVICE)
            logits = model(pixel_values=x).logits
            logits = F.interpolate(logits, size=y.shape[-2:], mode='bilinear', align_corners=False)
            probs = torch.sigmoid(logits[:, 1:2] - logits[:, 0:1])
            for b in range(x.size(0)):
                m = d.compute_edge_metrics(probs[b, 0].cpu().numpy(), y[b, 0].cpu().numpy())
                ious.append(m['iou']); f1s.append(m['f1'])
    history_D['train'].append(train_loss)
    history_D['val_iou'].append(float(np.mean(ious))); history_D['val_f1'].append(float(np.mean(f1s)))
    print(f'epoch {epoch}: train={train_loss:.3f}  val IoU={np.mean(ious):.3f}  val F1={np.mean(f1s):.3f}')

torch.save({'model': model.state_dict(), 'history': history_D}, CKPT)
print('saved', CKPT)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(history_D['train']); ax[0].set_title('train loss (BCE+Dice)')
ax[1].plot(history_D['val_iou'], label='IoU'); ax[1].plot(history_D['val_f1'], label='F1'); ax[1].legend(); ax[1].set_title('val metrics')
plt.show()